In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from xgboost import XGBClassifier

print("Libraries imported successfully!")

Libraries imported successfully!


### Define Project Paths

In [2]:
BASE_PATH = r"C:\Users\Rounak\OneDrive\Documents\nids"

PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")
MODEL_PATH = os.path.join(BASE_PATH, "models")

X_TRAIN_PATH = os.path.join(PROCESSED_PATH, "X_train.csv")
X_TEST_PATH = os.path.join(PROCESSED_PATH, "X_test.csv")

Y_TRAIN_PATH = os.path.join(PROCESSED_PATH, "y_train.csv")
Y_TEST_PATH = os.path.join(PROCESSED_PATH, "y_test.csv")

LABEL_MAPPING_PATH = os.path.join(PROCESSED_PATH, "label_mapping.csv")

print("Processed data path:")
print(PROCESSED_PATH)

print("\nChecking required files...\n")

required_files = [
    X_TRAIN_PATH,
    X_TEST_PATH,
    Y_TRAIN_PATH,
    Y_TEST_PATH,
    LABEL_MAPPING_PATH
]

for file_path in required_files:
    print(os.path.basename(file_path), "->", os.path.exists(file_path))

Processed data path:
C:\Users\Rounak\OneDrive\Documents\nids\data\processed

Checking required files...

X_train.csv -> True
X_test.csv -> True
y_train.csv -> True
y_test.csv -> True
label_mapping.csv -> True


### Load Feature Data Correctly

In [3]:
header=None

In [4]:
X_train = pd.read_csv(X_TRAIN_PATH, header=None)
X_test = pd.read_csv(X_TEST_PATH, header=None)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nFirst few X_train columns:")
print(X_train.columns[:10].tolist())

print("\nNumber of training features:", X_train.shape[1])
print("Number of testing features:", X_test.shape[1])

X_train shape: (13106460, 77)
X_test shape: (3276616, 77)

First few X_train columns:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Number of training features: 77
Number of testing features: 77


### Load Target Labels Safely

In [5]:
y_train = pd.read_csv(Y_TRAIN_PATH)
y_test = pd.read_csv(Y_TEST_PATH)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\ny_train columns:")
print(y_train.columns.tolist())

print("\ny_test columns:")
print(y_test.columns.tolist())

print("\nFirst 5 training labels:")
print(y_train.head())

print("\nFirst 5 testing labels:")
print(y_test.head())

y_train shape: (13106459, 1)
y_test shape: (3276615, 1)

y_train columns:
['Label_Encoded']

y_test columns:
['Label_Encoded']

First 5 training labels:
   Label_Encoded
0              0
1              0
2              0
3              0
4              0

First 5 testing labels:
   Label_Encoded
0              0
1              0
2              0
3              0
4              0


### Extract and Clean the Numerical Labels

In [6]:
# Extract the first column from both target files
y_train = y_train.iloc[:, 0]
y_test = y_test.iloc[:, 0]

# Convert safely to numeric
y_train = pd.to_numeric(y_train, errors="coerce")
y_test = pd.to_numeric(y_test, errors="coerce")

print("Missing values in y_train:", y_train.isna().sum())
print("Missing values in y_test:", y_test.isna().sum())

# Remove missing values if any exist
y_train = y_train.dropna().astype(int)
y_test = y_test.dropna().astype(int)

print("\ny_train unique labels:")
print(sorted(y_train.unique()))

print("\ny_test unique labels:")
print(sorted(y_test.unique()))

print("\nNumber of training classes:", y_train.nunique())
print("Number of testing classes:", y_test.nunique())

Missing values in y_train: 0
Missing values in y_test: 0

y_train unique labels:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

y_test unique labels:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of training classes: 19
Number of testing classes: 19


### Verify Feature and Label Row Alignment

In [7]:
print("X_train rows:", len(X_train))
print("y_train rows:", len(y_train))

print("\nX_test rows:", len(X_test))
print("y_test rows:", len(y_test))

print("\nTraining data aligned:", len(X_train) == len(y_train))
print("Testing data aligned:", len(X_test) == len(y_test))

X_train rows: 13106460
y_train rows: 13106459

X_test rows: 3276616
y_test rows: 3276615

Training data aligned: False
Testing data aligned: False


### Reload

In [9]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# PROJECT PATHS
# --------------------------------------------------

BASE_PATH = Path(r"C:\Users\Rounak\OneDrive\Documents\nids")

X_TRAIN_PATH = BASE_PATH / "data" / "processed" / "X_train.csv"
X_TEST_PATH = BASE_PATH / "data" / "processed" / "X_test.csv"

Y_TRAIN_PATH = BASE_PATH / "data" / "processed" / "y_train.csv"
Y_TEST_PATH = BASE_PATH / "data" / "processed" / "y_test.csv"


# --------------------------------------------------
# CHECK FILES
# --------------------------------------------------

print("X_train exists:", X_TRAIN_PATH.exists())
print("X_test exists :", X_TEST_PATH.exists())
print("y_train exists:", Y_TRAIN_PATH.exists())
print("y_test exists :", Y_TEST_PATH.exists())

X_train exists: True
X_test exists : True
y_train exists: True
y_test exists : True


In [10]:
print("================================================")
print("X_TRAIN PREVIEW")
print("================================================")

X_train_sample = pd.read_csv(X_TRAIN_PATH, nrows=5)

print("Shape of sample:")
print(X_train_sample.shape)

print("\nColumns:")
print(X_train_sample.columns.tolist())

print("\nData:")
print(X_train_sample)


print("\n================================================")
print("Y_TRAIN PREVIEW")
print("================================================")

y_train_sample = pd.read_csv(Y_TRAIN_PATH, nrows=10)

print("Shape of sample:")
print(y_train_sample.shape)

print("\nColumns:")
print(y_train_sample.columns.tolist())

print("\nData:")
print(y_train_sample)

X_TRAIN PREVIEW
Shape of sample:
(5, 77)

Columns:
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76']

Data:
          0         1         2         3         4         5         6  \
0 -0.514962 -0.028937 -0.015320 -0.015640 -0.015939 -0.007559 -0.395518   
1  2.222816 -0.028962 -0.014673 -0.017884 -0.017036 -0.007714 -0.558887   
2 -0.495405  0.208531 -0.001088  0.035979  0.006649  0.001762  2.100651   
3  2.094143 -0.020338 -0.012732 -0.011151 -0.001960 -0.007340  1.687426   
4 -0.495405  0.078901 -0.009498  0.015780  0.010761 -0.004060  2.292849   

          7         8         9  ...  

In [11]:
print("================================================")
print("X_TEST PREVIEW")
print("================================================")

X_test_sample = pd.read_csv(X_TEST_PATH, nrows=5)

print("Shape of sample:")
print(X_test_sample.shape)

print("\nColumns:")
print(X_test_sample.columns.tolist())

print("\nData:")
print(X_test_sample)


print("\n================================================")
print("Y_TEST PREVIEW")
print("================================================")

y_test_sample = pd.read_csv(Y_TEST_PATH, nrows=10)

print("Shape of sample:")
print(y_test_sample.shape)

print("\nColumns:")
print(y_test_sample.columns.tolist())

print("\nData:")
print(y_test_sample)

X_TEST PREVIEW
Shape of sample:
(5, 77)

Columns:
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76']

Data:
          0         1         2         3         4         5         6  \
0 -0.495405  0.093224 -0.010791 -0.004418  0.000845 -0.003721  1.312640   
1  2.023839 -0.028963 -0.014673 -0.017884 -0.016536 -0.007714 -0.484410   
2 -0.495405  0.206715 -0.001735  0.033735 -0.007249 -0.001331  0.262759   
3 -0.514962 -0.028959 -0.015320 -0.015640 -0.016359 -0.007567 -0.457983   
4 -0.514962 -0.028958 -0.015320 -0.015640 -0.016455 -0.007622 -0.472397   

          7         8         9  ...   

### inspect the extra row situation

In [12]:
print("=" * 60)
print("LAST ROW COMPARISON")
print("=" * 60)

print("\nX_train last 3 rows:")
print(X_train.tail(3))

print("\ny_train last 3 rows:")
print(y_train.tail(3))

print("\nX_test last 3 rows:")
print(X_test.tail(3))

print("\ny_test last 3 rows:")
print(y_test.tail(3))

LAST ROW COMPARISON

X_train last 3 rows:
                0         1         2         3         4         5   \
13106457 -0.514962 -0.028917 -0.014673 -0.013396 -0.015681 -0.007594   
13106458 -0.513608 -0.017450 -0.013379 -0.008907 -0.015472 -0.007474   
13106459 -0.495405 -0.028962 -0.014026 -0.017884 -0.015794 -0.007714   

                6         7         8         9   ...        67        68  \
13106457 -0.457983  0.904231 -0.164474 -0.507367  ... -0.012788  0.001036   
13106458 -0.325847 -0.367157 -0.349433 -0.207473  ... -0.012788  0.001063   
13106459 -0.448373 -0.367157 -0.334671 -0.362310  ... -0.012788  0.001063   

                69        70        71       72        73        74        75  \
13106457 -0.073909 -0.061178 -0.086023 -0.05878 -0.024411 -0.002203 -0.011044   
13106458 -0.073909 -0.061178 -0.086023 -0.05878 -0.024411 -0.002203 -0.011044   
13106459 -0.073909 -0.061178 -0.086023 -0.05878 -0.024411 -0.002203 -0.011044   

                76  
13106457 -0.09

### ALLIGNMENT

In [14]:

print("=" * 60)
print("ALIGNING FEATURES AND LABELS")
print("=" * 60)

# Slice without creating a deep copy
X_train = X_train.iloc[:len(y_train)]
X_test = X_test.iloc[:len(y_test)]

print("\nAfter alignment:")

print("\nX_train rows:", len(X_train))
print("y_train rows:", len(y_train))

print("\nX_test rows:", len(X_test))
print("y_test rows:", len(y_test))

print("\nTraining data aligned:", len(X_train) == len(y_train))
print("Testing data aligned:", len(X_test) == len(y_test))

print("\nFinal shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

ALIGNING FEATURES AND LABELS

After alignment:

X_train rows: 13106459
y_train rows: 13106459

X_test rows: 3276615
y_test rows: 3276615

Training data aligned: True
Testing data aligned: True

Final shapes:
X_train: (13106459, 77)
y_train: (13106459,)

X_test: (3276615, 77)
y_test: (3276615,)


### Final Label Verification

In [15]:
print("=" * 60)
print("FINAL LABEL VERIFICATION")
print("=" * 60)

# Unique labels
train_labels = sorted(y_train.unique())
test_labels = sorted(y_test.unique())

print("\nTraining labels:")
print(train_labels)

print("\nNumber of training classes:")
print(len(train_labels))

print("\nTesting labels:")
print(test_labels)

print("\nNumber of testing classes:")
print(len(test_labels))

# Check compatibility
print("\nAll training labels present in testing data:")
print(set(train_labels).issubset(set(test_labels)))

print("\nAll testing labels present in training data:")
print(set(test_labels).issubset(set(train_labels)))

# Label distributions
print("\n" + "=" * 60)
print("TRAINING LABEL DISTRIBUTION")
print("=" * 60)
print(y_train.value_counts().sort_index())

print("\n" + "=" * 60)
print("TESTING LABEL DISTRIBUTION")
print("=" * 60)
print(y_test.value_counts().sort_index())

FINAL LABEL VERIFICATION

Training labels:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of training classes:
19

Testing labels:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of testing classes:
19

All training labels present in testing data:
True

All testing labels present in training data:
True

TRAINING LABEL DISTRIBUTION
Label_Encoded
0     11521130
1       117929
2         1632
3          705
4       219920
5         1384
6       102413
7       460645
8        41353
9       335401
10        4305
11       12278
12        4858
13           9
14      

# Documentation – Dataset Row Alignment and Label Compatibility Fix

During the model verification process, it was discovered that the feature datasets and their corresponding label datasets were not perfectly aligned. X_train contained 13,106,460 rows, while y_train contained 13,106,459 rows. Similarly, X_test contained 3,276,616 rows, while y_test contained 3,276,615 rows. Inspection of the dataset indices confirmed that the feature files contained exactly one additional row at the end. The feature datasets were therefore safely truncated to the length of their corresponding label datasets without creating unnecessary deep copies, which was important due to the large size of the dataset and memory limitations. After alignment, the final dataset shapes became X_train (13,106,459 × 77), y_train (13,106,459), X_test (3,276,615 × 77), and y_test (3,276,615), confirming that every feature row now has a corresponding label.

A final label verification was then performed to investigate the previously observed mismatch between the saved model and the dataset. Both the training and testing label datasets contained 19 unique encoded classes, ranging from 0 to 18, and all classes present in the training dataset were also present in the testing dataset. This confirmed that the dataset itself contains a complete 19-class intrusion detection problem. However, the previously saved XGBoost model and label encoder supported only 13 classes, explaining why the old model was unable to correctly classify several attack categories and showed a strong tendency to predict only the BENIGN class. The row alignment issue has now been resolved, and the dataset has been verified to contain 77 input features and 19 compatible classes in both training and testing data. The next step is therefore to create a new 19-class label encoding setup and retrain the XGBoost model using the corrected and fully aligned dataset.

### Import Required Libraries

In [16]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

### FINAL DATASET SAFETY TEST

In [17]:
print("=" * 60)
print("FINAL DATASET SAFETY CHECK")
print("=" * 60)

print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTraining rows aligned:",
      len(X_train) == len(y_train))

print("Testing rows aligned:",
      len(X_test) == len(y_test))

print("\nTraining classes:")
print(sorted(y_train.unique()))

print("\nNumber of training classes:",
      y_train.nunique())

print("\nTesting classes:")
print(sorted(y_test.unique()))

print("\nNumber of testing classes:",
      y_test.nunique())

print("\nFeature count:", X_train.shape[1])

print("\nFeature columns match:")
print(list(X_train.columns) == list(X_test.columns))

FINAL DATASET SAFETY CHECK

X_train shape: (13106459, 77)
y_train shape: (13106459,)

X_test shape: (3276615, 77)
y_test shape: (3276615,)

Training rows aligned: True
Testing rows aligned: True

Training classes:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of training classes: 19

Testing classes:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of testing classes: 19

Feature count: 77

Feature columns match:
True


### CHECK EXTREME CLASS IMBALANCE

In [18]:
print("=" * 60)
print("CLASS DISTRIBUTION CHECK")
print("=" * 60)

class_counts = y_train.value_counts().sort_index()

print("\nTraining class distribution:")
print(class_counts)

print("\nSmallest class size:",
      class_counts.min())

print("\nLargest class size:",
      class_counts.max())

print("\nImbalance ratio:")
print(round(class_counts.max() / class_counts.min(), 2))

CLASS DISTRIBUTION CHECK

Training class distribution:
Label_Encoded
0     11521130
1       117929
2         1632
3          705
4       219920
5         1384
6       102413
7       460645
8        41353
9       335401
10        4305
11       12278
12        4858
13           9
14      117225
15          29
16       87217
17          85
18       77941
Name: count, dtype: int64

Smallest class size: 9

Largest class size: 11521130

Imbalance ratio:
1280125.56


In [19]:
TRAINING_SAMPLE_SIZE = 1000000

all_indices = np.arange(len(X_train))

sample_indices, _ = train_test_split(
    all_indices,
    train_size=TRAINING_SAMPLE_SIZE,
    stratify=y_train,
    random_state=42
)

X_train_sample = X_train.iloc[sample_indices].copy()
y_train_sample = y_train.iloc[sample_indices].copy()

print("=" * 60)
print("TRAINING SAMPLE CREATED")
print("=" * 60)

print("\nX_train_sample shape:")
print(X_train_sample.shape)

print("\ny_train_sample shape:")
print(y_train_sample.shape)

print("\nClasses in training sample:")
print(sorted(y_train_sample.unique()))

print("\nNumber of classes:")
print(y_train_sample.nunique())

print("\nClass distribution:")
print(y_train_sample.value_counts().sort_index())

TRAINING SAMPLE CREATED

X_train_sample shape:
(1000000, 77)

y_train_sample shape:
(1000000,)

Classes in training sample:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of classes:
19

Class distribution:
Label_Encoded
0     879042
1       8998
2        125
3         54
4      16780
5        106
6       7814
7      35146
8       3155
9      25590
10       328
11       937
12       371
13         1
14      8944
15         2
16      6654
17         6
18      5947
Name: count, dtype: int64


A stratified sample of one million records was successfully created from the corrected training dataset while preserving all 19 attack classes. However, inspection of the sampled class distribution revealed severe class imbalance, with the BENIGN class representing the majority of the data and several rare attack classes containing only one to six samples in the training subset. Although all classes were technically preserved, such extremely small sample counts are insufficient for reliable model learning. Therefore, instead of directly training the XGBoost model on the stratified sample, a controlled sampling strategy will be applied to retain all available rare attack samples while limiting the dominance of majority classes. This approach aims to provide a more meaningful representation of attack categories during model training while remaining computationally manageable.

In [20]:
# Remove the previous stratified sample to free memory

del X_train_sample
del y_train_sample

print("Previous 1 million row stratified sample removed.")

Previous 1 million row stratified sample removed.


### Define the Controlled Sampling Limits

In [21]:
# Controlled sampling limits

BENIGN_LIMIT = 200000
ATTACK_CLASS_LIMIT = 50000

print("BENIGN sample limit:", BENIGN_LIMIT)
print("Maximum samples per attack class:", ATTACK_CLASS_LIMIT)

BENIGN sample limit: 200000
Maximum samples per attack class: 50000


### Controlled Sample

In [23]:
print("=" * 60)
print("CREATING CONTROLLED TRAINING DATASET")
print("=" * 60)

selected_indices = []

# Process every class from 0 to 18
for label in sorted(y_train.unique()):

    # Find positions belonging to this class
    class_indices = np.where(
        y_train.to_numpy() == label
    )[0]

    # BENIGN class gets its own limit
    if label == 0:
        sample_limit = BENIGN_LIMIT
    else:
        sample_limit = ATTACK_CLASS_LIMIT

    # Keep all samples if the class is smaller than the limit
    if len(class_indices) <= sample_limit:
        selected = class_indices

        print(
            f"Class {label}: {len(class_indices)} available "
            f"-> keeping ALL {len(selected)}"
        )

    else:
        selected = np.random.choice(
            class_indices,
            size=sample_limit,
            replace=False
        )

        print(
            f"Class {label}: {len(class_indices)} available "
            f"-> sampled {len(selected)}"
        )

    selected_indices.extend(selected)

print("\nTotal selected rows before shuffling:",
      len(selected_indices))

CREATING CONTROLLED TRAINING DATASET
Class 0: 11521130 available -> sampled 200000
Class 1: 117929 available -> sampled 50000
Class 2: 1632 available -> keeping ALL 1632
Class 3: 705 available -> keeping ALL 705
Class 4: 219920 available -> sampled 50000
Class 5: 1384 available -> keeping ALL 1384
Class 6: 102413 available -> sampled 50000
Class 7: 460645 available -> sampled 50000
Class 8: 41353 available -> keeping ALL 41353
Class 9: 335401 available -> sampled 50000
Class 10: 4305 available -> keeping ALL 4305
Class 11: 12278 available -> keeping ALL 12278
Class 12: 4858 available -> keeping ALL 4858
Class 13: 9 available -> keeping ALL 9
Class 14: 117225 available -> sampled 50000
Class 15: 29 available -> keeping ALL 29
Class 16: 87217 available -> sampled 50000
Class 17: 85 available -> keeping ALL 85
Class 18: 77941 available -> sampled 50000

Total selected rows before shuffling: 666638


### Shuffle the selected rows

In [24]:
# Convert to NumPy array
selected_indices = np.array(selected_indices)

# Shuffle the selected row positions
np.random.seed(42)
np.random.shuffle(selected_indices)

print("Selected indices shuffled successfully.")

Selected indices shuffled successfully.


### Creating the final controlled training dataset

In [25]:
print("Creating final controlled training dataset...")

X_train_controlled = (
    X_train.iloc[selected_indices]
    .reset_index(drop=True)
)

y_train_controlled = (
    y_train.iloc[selected_indices]
    .reset_index(drop=True)
)

print("\nControlled dataset created successfully.")

print("\nX_train_controlled shape:")
print(X_train_controlled.shape)

print("\ny_train_controlled shape:")
print(y_train_controlled.shape)

Creating final controlled training dataset...

Controlled dataset created successfully.

X_train_controlled shape:
(666638, 77)

y_train_controlled shape:
(666638,)


### Verify the New Class Distribution

In [26]:
print("=" * 60)
print("CONTROLLED TRAINING CLASS DISTRIBUTION")
print("=" * 60)

controlled_counts = (
    y_train_controlled
    .value_counts()
    .sort_index()
)

print(controlled_counts)

print("\nNumber of classes:",
      y_train_controlled.nunique())

print("\nClasses present:")
print(sorted(y_train_controlled.unique()))

print("\nSmallest class size:",
      controlled_counts.min())

print("\nLargest class size:",
      controlled_counts.max())

CONTROLLED TRAINING CLASS DISTRIBUTION
Label_Encoded
0     200000
1      50000
2       1632
3        705
4      50000
5       1384
6      50000
7      50000
8      41353
9      50000
10      4305
11     12278
12      4858
13         9
14     50000
15        29
16     50000
17        85
18     50000
Name: count, dtype: int64

Number of classes: 19

Classes present:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Smallest class size: 9

Largest class size: 200000


### FINAL Feature and Label Alignment Check

In [27]:
print("=" * 60)
print("FINAL CONTROLLED DATASET CHECK")
print("=" * 60)

print("\nFeature rows:",
      len(X_train_controlled))

print("Label rows:",
      len(y_train_controlled))

print("\nRows aligned:",
      len(X_train_controlled)
      == len(y_train_controlled))

print("\nNumber of features:",
      X_train_controlled.shape[1])

print("\nExpected number of classes: 19")

print("Actual number of classes:",
      y_train_controlled.nunique())

print("\nAll classes 0-18 present:")

expected_classes = set(range(19))
actual_classes = set(
    y_train_controlled.unique()
)

print(expected_classes == actual_classes)

FINAL CONTROLLED DATASET CHECK

Feature rows: 666638
Label rows: 666638

Rows aligned: True

Number of features: 77

Expected number of classes: 19
Actual number of classes: 19

All classes 0-18 present:
True


### Creating the model

In [28]:
from xgboost import XGBClassifier

print("=" * 60)
print("CREATING 19-CLASS XGBOOST MODEL")
print("=" * 60)

xgb_model_19 = XGBClassifier(
    objective="multi:softprob",
    num_class=19,

    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method="hist",

    eval_metric="mlogloss",

    random_state=42,
    n_jobs=-1
)

print("Model created successfully.")
print("Number of classes configured:", 19)
print("Number of estimators:", 200)

CREATING 19-CLASS XGBOOST MODEL
Model created successfully.
Number of classes configured: 19
Number of estimators: 200


### Training the model

In [29]:
print("=" * 60)
print("STARTING MODEL TRAINING")
print("=" * 60)

xgb_model_19.fit(
    X_train_controlled,
    y_train_controlled
)

print("\n" + "=" * 60)
print("MODEL TRAINING COMPLETED")
print("=" * 60)

STARTING MODEL TRAINING

MODEL TRAINING COMPLETED


### Save the model

In [30]:
import joblib
import os

# Create models directory if it does not exist
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save trained 19-class XGBoost model
MODEL_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_intrusion_model_19class.joblib"
)

joblib.dump(xgb_model_19, MODEL_PATH)

print("=" * 60)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 60)

print("\nModel path:")
print(MODEL_PATH)

print("\nNumber of classes:")
print(xgb_model_19.n_classes_)

print("\nNumber of features:")
print(xgb_model_19.n_features_in_)

MODEL SAVED SUCCESSFULLY

Model path:
models\xgboost_intrusion_model_19class.joblib

Number of classes:
19

Number of features:
77


After successfully training the 19-class XGBoost intrusion detection model, the trained model was saved using the Joblib serialization library. Saving the trained model ensures that the learned parameters and classification structure can be reused later without repeating the computationally expensive training process. The model was stored separately from the preprocessing artifacts and retains support for all 19 encoded intrusion classes and the 77 input features used during training.

### Creation of controlled test dataset

In [31]:
import pandas as pd
import numpy as np

print("=" * 60)
print("CREATING CONTROLLED TEST DATASET")
print("=" * 60)

MAX_TEST_SAMPLES_PER_CLASS = 10000

# Make sure indices are positional and aligned
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

selected_indices = []

# Select samples class by class
for label in sorted(y_test_reset.unique()):
    
    class_indices = y_test_reset[
        y_test_reset == label
    ].index
    
    if len(class_indices) > MAX_TEST_SAMPLES_PER_CLASS:
        
        sampled_indices = class_indices.to_series().sample(
            n=MAX_TEST_SAMPLES_PER_CLASS,
            random_state=42
        ).index
        
    else:
        
        sampled_indices = class_indices
    
    selected_indices.extend(sampled_indices)

# Sort indices
selected_indices = sorted(selected_indices)

# Create controlled test dataset
X_test_controlled = X_test_reset.iloc[selected_indices].reset_index(drop=True)

y_test_controlled = y_test_reset.iloc[
    selected_indices
].reset_index(drop=True)

print("\nControlled test feature shape:")
print(X_test_controlled.shape)

print("\nControlled test label shape:")
print(y_test_controlled.shape)

print("\nRows aligned:")
print(len(X_test_controlled) == len(y_test_controlled))

print("\nNumber of features:")
print(X_test_controlled.shape[1])

print("\nClasses present:")
print(sorted(y_test_controlled.unique()))

print("\nNumber of classes:")
print(y_test_controlled.nunique())

print("\nControlled test class distribution:")
print(y_test_controlled.value_counts().sort_index())

CREATING CONTROLLED TEST DATASET

Controlled test feature shape:
(106320, 77)

Controlled test label shape:
(106320,)

Rows aligned:
True

Number of features:
77

Classes present:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of classes:
19

Controlled test class distribution:
Label_Encoded
0     10000
1     10000
2       408
3       176
4     10000
5       346
6     10000
7     10000
8     10000
9     10000
10     1076
11     3069
12     1215
13        2
14    10000
15        7
16    10000
17       21
18    10000
Name: count, dtype: int64


### Verify Model Compatibility Before Prediction

In [32]:
print("=" * 60)
print("FINAL MODEL AND TEST COMPATIBILITY CHECK")
print("=" * 60)

print("\nModel expected features:")
print(xgb_model_19.n_features_in_)

print("\nTest dataset features:")
print(X_test_controlled.shape[1])

print("\nFeature count compatible:")
print(
    xgb_model_19.n_features_in_ ==
    X_test_controlled.shape[1]
)

print("\nModel classes:")
print(xgb_model_19.classes_)

print("\nNumber of model classes:")
print(len(xgb_model_19.classes_))

print("\nClasses in controlled test data:")
print(sorted(y_test_controlled.unique()))

print("\nNumber of test classes:")
print(y_test_controlled.nunique())

print("\nAll test classes supported by model:")
print(
    set(y_test_controlled.unique()).issubset(
        set(xgb_model_19.classes_)
    )
)

print("\nFeature rows and label rows aligned:")
print(
    len(X_test_controlled) ==
    len(y_test_controlled)
)

FINAL MODEL AND TEST COMPATIBILITY CHECK

Model expected features:
77

Test dataset features:
77

Feature count compatible:
True

Model classes:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]

Number of model classes:
19

Classes in controlled test data:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of test classes:
19

All test classes supported by model:
True

Feature rows and label rows aligned:
True


### Generating predictions

In [33]:
print("=" * 60)
print("GENERATING MODEL PREDICTIONS")
print("=" * 60)

# Generate predictions
y_pred_controlled = xgb_model_19.predict(
    X_test_controlled
)

print("\nPrediction completed successfully.")

print("\nNumber of predictions:")
print(len(y_pred_controlled))

print("\nNumber of test samples:")
print(len(y_test_controlled))

print("\nPredictions aligned with test labels:")
print(
    len(y_pred_controlled) ==
    len(y_test_controlled)
)

print("\nPredicted classes:")
print(
    sorted(np.unique(y_pred_controlled))
)

print("\nNumber of predicted classes:")
print(
    len(np.unique(y_pred_controlled))
)

print("\nPredicted class distribution:")
print(
    pd.Series(y_pred_controlled)
    .value_counts()
    .sort_index()
)

GENERATING MODEL PREDICTIONS

Prediction completed successfully.

Number of predictions:
106320

Number of test samples:
106320

Predictions aligned with test labels:
True

Predicted classes:
[np.int64(0), np.int64(1), np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(16), np.int64(17), np.int64(18)]

Number of predicted classes:
14

Predicted class distribution:
0     106087
1         23
4         25
6         24
7         27
8          9
9         35
10         1
11         2
12         1
14        22
16        37
17         1
18        26
Name: count, dtype: int64


After confirming complete compatibility between the trained XGBoost model and the controlled test dataset, predictions were generated for all 106,320 selected test samples. The model received the same 77-feature structure used during training, while the test labels remained positionally aligned with their corresponding feature rows. The predicted class distribution was examined to verify that the model was capable of producing multiple intrusion categories rather than repeating a single dominant class. This evaluation directly addresses the earlier issue where the previously used model predicted only the BENIGN class because of inconsistencies in model class support and dataset compatibility.

### Evaluate Before Changing Anything

In [34]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("CONTROLLED TEST EVALUATION")
print("=" * 60)

# Overall accuracy
accuracy = accuracy_score(
    y_test_controlled,
    y_pred_controlled
)

print("\nOverall Accuracy:")
print(round(accuracy, 6))

print("\nNumber of correct predictions:")
correct = np.sum(
    y_test_controlled.to_numpy() ==
    y_pred_controlled
)

print(correct)

print("\nTotal predictions:")
print(len(y_test_controlled))

print("\nClassification Report:")
print(
    classification_report(
        y_test_controlled,
        y_pred_controlled,
        labels=list(range(19)),
        zero_division=0
    )
)

CONTROLLED TEST EVALUATION

Overall Accuracy:
0.094074

Number of correct predictions:
10002

Total predictions:
106320

Classification Report:
              precision    recall  f1-score   support

           0       0.09      1.00      0.17     10000
           1       0.04      0.00      0.00     10000
           2       0.00      0.00      0.00       408
           3       0.00      0.00      0.00       176
           4       0.08      0.00      0.00     10000
           5       0.00      0.00      0.00       346
           6       0.17      0.00      0.00     10000
           7       0.04      0.00      0.00     10000
           8       0.11      0.00      0.00     10000
           9       0.14      0.00      0.00     10000
          10       0.00      0.00      0.00      1076
          11       0.00      0.00      0.00      3069
          12       0.00      0.00      0.00      1215
          13       0.00      0.00      0.00         2
          14       0.05      0.00      0.00  

### Create a Controlled Training Evaluation Sample

In [35]:
print("=" * 60)
print("CREATING TRAINING EVALUATION SAMPLE")
print("=" * 60)

MAX_TRAIN_EVAL_PER_CLASS = 5000

# Reset indices to ensure positional alignment
X_train_eval_source = X_train_controlled.reset_index(drop=True)
y_train_eval_source = y_train_controlled.reset_index(drop=True)

selected_train_indices = []

# Select up to 5,000 samples from every class
for label in sorted(y_train_eval_source.unique()):

    class_indices = y_train_eval_source[
        y_train_eval_source == label
    ].index

    if len(class_indices) > MAX_TRAIN_EVAL_PER_CLASS:

        sampled_indices = class_indices.to_series().sample(
            n=MAX_TRAIN_EVAL_PER_CLASS,
            random_state=42
        ).index

    else:
        sampled_indices = class_indices

    selected_train_indices.extend(sampled_indices)

# Sort indices to preserve dataset order
selected_train_indices = sorted(selected_train_indices)

# Create evaluation sample
X_train_eval = X_train_eval_source.iloc[
    selected_train_indices
].reset_index(drop=True)

y_train_eval = y_train_eval_source.iloc[
    selected_train_indices
].reset_index(drop=True)

print("\nTraining evaluation feature shape:")
print(X_train_eval.shape)

print("\nTraining evaluation label shape:")
print(y_train_eval.shape)

print("\nRows aligned:")
print(len(X_train_eval) == len(y_train_eval))

print("\nNumber of features:")
print(X_train_eval.shape[1])

print("\nClasses present:")
print(sorted(y_train_eval.unique()))

print("\nNumber of classes:")
print(y_train_eval.nunique())

print("\nTraining evaluation class distribution:")
print(y_train_eval.value_counts().sort_index())

CREATING TRAINING EVALUATION SAMPLE

Training evaluation feature shape:
(68007, 77)

Training evaluation label shape:
(68007,)

Rows aligned:
True

Number of features:
77

Classes present:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of classes:
19

Training evaluation class distribution:
Label_Encoded
0     5000
1     5000
2     1632
3      705
4     5000
5     1384
6     5000
7     5000
8     5000
9     5000
10    4305
11    5000
12    4858
13       9
14    5000
15      29
16    5000
17      85
18    5000
Name: count, dtype: int64


### Predict on the Training Evaluation Sample

In [36]:
print("=" * 60)
print("TRAINING DATA SANITY CHECK")
print("=" * 60)

# Generate predictions on data seen during training
y_pred_train_eval = xgb_model_19.predict(
    X_train_eval
)

print("\nPrediction completed.")

print("\nNumber of predictions:")
print(len(y_pred_train_eval))

print("\nPredicted classes:")
print(sorted(np.unique(y_pred_train_eval)))

print("\nNumber of predicted classes:")
print(len(np.unique(y_pred_train_eval)))

print("\nPredicted class distribution:")
print(
    pd.Series(y_pred_train_eval)
    .value_counts()
    .sort_index()
)

TRAINING DATA SANITY CHECK

Prediction completed.

Number of predictions:
68007

Predicted classes:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

Number of predicted classes:
19

Predicted class distribution:
0     67026
1        58
2        61
3        54
4        53
5        70
6        51
7        59
8        56
9        61
10       83
11       72
12       93
13        2
14       58
15        6
16       72
17       23
18       49
Name: count, dtype: int64


### Calculate Training Performance

In [37]:
from sklearn.metrics import accuracy_score, classification_report

print("=" * 60)
print("TRAINING DATA PERFORMANCE")
print("=" * 60)

train_accuracy = accuracy_score(
    y_train_eval,
    y_pred_train_eval
)

correct_train_predictions = np.sum(
    y_train_eval.to_numpy() == y_pred_train_eval
)

print("\nTraining Evaluation Accuracy:")
print(round(train_accuracy, 6))

print("\nCorrect predictions:")
print(correct_train_predictions)

print("\nTotal samples:")
print(len(y_train_eval))

print("\nClassification Report:")
print(
    classification_report(
        y_train_eval,
        y_pred_train_eval,
        labels=list(range(19)),
        zero_division=0
    )
)

TRAINING DATA PERFORMANCE

Training Evaluation Accuracy:
0.087432

Correct predictions:
5946

Total samples:
68007

Classification Report:
              precision    recall  f1-score   support

           0       0.07      1.00      0.14      5000
           1       0.95      0.01      0.02      5000
           2       1.00      0.04      0.07      1632
           3       1.00      0.08      0.14       705
           4       0.96      0.01      0.02      5000
           5       1.00      0.05      0.10      1384
           6       0.92      0.01      0.02      5000
           7       0.88      0.01      0.02      5000
           8       1.00      0.01      0.02      5000
           9       0.92      0.01      0.02      5000
          10       1.00      0.02      0.04      4305
          11       1.00      0.01      0.03      5000
          12       1.00      0.02      0.04      4858
          13       1.00      0.22      0.36         9
          14       0.98      0.01      0.02      5

### Inspect the model

In [38]:
print("=" * 60)
print("XGBOOST MODEL CONFIGURATION CHECK")
print("=" * 60)

print("\nModel parameters:")
print(xgb_model_19.get_params())

print("\nConfigured number of classes:")
print(xgb_model_19.n_classes_)

print("\nModel classes:")
print(xgb_model_19.classes_)

print("\nNumber of trees / estimators:")
print(xgb_model_19.n_estimators)

XGBOOST MODEL CONFIGURATION CHECK

Model parameters:
{'objective': 'multi:softprob', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'mlogloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 8, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.8, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None, 'num_class': 19}

Configur

### Check if model built useful tress

In [39]:
print("=" * 60)
print("MODEL TREE CHECK")
print("=" * 60)

booster = xgb_model_19.get_booster()

print("\nNumber of boosted rounds:")
print(booster.num_boosted_rounds())

print("\nFeature importance:")

importance = booster.get_score(importance_type="weight")

print("Number of features used by model:")
print(len(importance))

print("\nTop 20 most used features:")

sorted_importance = sorted(
    importance.items(),
    key=lambda x: x[1],
    reverse=True
)

for feature, score in sorted_importance[:20]:
    print(feature, ":", score)

MODEL TREE CHECK

Number of boosted rounds:
200

Feature importance:
Number of features used by model:
69

Top 20 most used features:
19 : 18574.0
24 : 17614.0
0 : 14258.0
14 : 12332.0
65 : 11901.0
1 : 11630.0
29 : 11234.0
18 : 9087.0
4 : 9058.0
17 : 8889.0
15 : 8669.0
22 : 8333.0
66 : 8202.0
36 : 7847.0
16 : 7726.0
40 : 7686.0
37 : 7668.0
20 : 7647.0
8 : 7508.0
25 : 7493.0


### Check prediction probabilities

In [40]:
print("=" * 60)
print("PREDICTION PROBABILITY CHECK")
print("=" * 60)

# Take a small sample
X_probability_sample = X_train_eval.iloc[:20]

# Get probabilities
probabilities = xgb_model_19.predict_proba(
    X_probability_sample
)

# Display predicted probabilities
probability_df = pd.DataFrame(
    probabilities,
    columns=[f"Class_{i}" for i in range(19)]
)

print("\nProbability shape:")
print(probability_df.shape)

print("\nFirst 10 probability predictions:")
print(probability_df.head(10))

print("\nPredicted class:")
print(
    np.argmax(probabilities, axis=1)
)

print("\nTrue class:")
print(
    y_train_eval.iloc[:20].to_numpy()
)

PREDICTION PROBABILITY CHECK

Probability shape:
(20, 19)

First 10 probability predictions:
    Class_0   Class_1   Class_2   Class_3   Class_4   Class_5   Class_6  \
0  0.278661  0.067613  0.002211  0.002869  0.066736  0.001835  0.082172   
1  0.300075  0.070379  0.001827  0.000347  0.070213  0.000757  0.057249   
2  0.311741  0.047567  0.000206  0.000216  0.078390  0.000827  0.074360   
3  0.386019  0.065806  0.000935  0.000280  0.072511  0.001627  0.086057   
4  0.295393  0.084464  0.002504  0.002052  0.076862  0.003171  0.070174   
5  0.297949  0.070722  0.005086  0.000269  0.073546  0.001383  0.074641   
6  0.295243  0.064714  0.001620  0.001144  0.074796  0.001552  0.066549   
7  0.295696  0.076970  0.001843  0.003818  0.080789  0.002596  0.075102   
8  0.304306  0.076906  0.002140  0.000871  0.076657  0.001474  0.072715   
9  0.294515  0.072694  0.002185  0.000822  0.075748  0.001147  0.072594   

    Class_7   Class_8   Class_9  Class_10  Class_11  Class_12      Class_13  \
0 

### The decisive experiment

We will now create a perfectly balanced debugging dataset.

In [41]:
import pandas as pd
import numpy as np

print("=" * 60)
print("CREATING BALANCED DEBUGGING DATASET")
print("=" * 60)

# Classes with extremely small numbers of samples
excluded_classes = [13, 15, 17]

# Use all remaining classes
available_classes = [
    c for c in range(19)
    if c not in excluded_classes
]

# Number of samples per class
SAMPLES_PER_CLASS = 300

X_balanced_list = []
y_balanced_list = []

for class_label in available_classes:

    # Get indices belonging to this class
    class_indices = y_train[
        y_train == class_label
    ].index

    # Randomly select samples
    selected_indices = np.random.choice(
        class_indices,
        size=SAMPLES_PER_CLASS,
        replace=False
    )

    # Extract corresponding features and labels
    X_balanced_list.append(
        X_train.loc[selected_indices]
    )

    y_balanced_list.append(
        y_train.loc[selected_indices]
    )

# Combine all classes
X_train_balanced_debug = pd.concat(
    X_balanced_list
).reset_index(drop=True)

y_train_balanced_debug = pd.concat(
    y_balanced_list
).reset_index(drop=True)

print("\nBalanced feature shape:")
print(X_train_balanced_debug.shape)

print("\nBalanced label shape:")
print(y_train_balanced_debug.shape)

print("\nRows aligned:")
print(
    len(X_train_balanced_debug)
    ==
    len(y_train_balanced_debug)
)

print("\nClass distribution:")
print(
    y_train_balanced_debug
    .value_counts()
    .sort_index()
)

print("\nNumber of classes:")
print(
    y_train_balanced_debug.nunique()
)

CREATING BALANCED DEBUGGING DATASET

Balanced feature shape:
(4800, 77)

Balanced label shape:
(4800,)

Rows aligned:
True

Class distribution:
Label_Encoded
0     300
1     300
2     300
3     300
4     300
5     300
6     300
7     300
8     300
9     300
10    300
11    300
12    300
14    300
16    300
18    300
Name: count, dtype: int64

Number of classes:
16


In [42]:
print("=" * 60)
print("CREATING CORRECTED BALANCED DEBUGGING DATASET")
print("=" * 60)

import pandas as pd

debug_X_parts = []
debug_y_parts = []

samples_per_class = 300

for label in sorted(y_train_controlled.unique()):

    # Get indices belonging to the current class
    class_indices = y_train_controlled[
        y_train_controlled == label
    ].index

    # Use maximum available samples if fewer than 300 exist
    actual_samples = min(samples_per_class, len(class_indices))

    # Randomly select samples
    selected_indices = class_indices.to_series().sample(
        n=actual_samples,
        random_state=42
    ).values

    # Select corresponding features and labels
    debug_X_parts.append(
        X_train_controlled.loc[selected_indices]
    )

    debug_y_parts.append(
        y_train_controlled.loc[selected_indices]
    )


# Combine all classes
X_debug = pd.concat(debug_X_parts).reset_index(drop=True)
y_debug = pd.concat(debug_y_parts).reset_index(drop=True)


print("\nBalanced feature shape:")
print(X_debug.shape)

print("\nBalanced label shape:")
print(y_debug.shape)

print("\nRows aligned:")
print(len(X_debug) == len(y_debug))

print("\nClass distribution:")
print(y_debug.value_counts().sort_index())

print("\nNumber of classes:")
print(y_debug.nunique())

print("\nClasses present:")
print(sorted(y_debug.unique()))

CREATING CORRECTED BALANCED DEBUGGING DATASET

Balanced feature shape:
(4923, 77)

Balanced label shape:
(4923,)

Rows aligned:
True

Class distribution:
Label_Encoded
0     300
1     300
2     300
3     300
4     300
5     300
6     300
7     300
8     300
9     300
10    300
11    300
12    300
13      9
14    300
15     29
16    300
17     85
18    300
Name: count, dtype: int64

Number of classes:
19

Classes present:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]


# CRITICAL DEBUG TEST

In [43]:
print("=" * 60)
print("BALANCED DEBUGGING PREDICTION TEST")
print("=" * 60)

# Generate predictions
debug_predictions = xgb_model_19.predict(X_debug)

print("\nNumber of predictions:")
print(len(debug_predictions))

print("\nNumber of debugging samples:")
print(len(y_debug))

print("\nPredictions aligned:")
print(len(debug_predictions) == len(y_debug))


# Convert predictions to integer labels
debug_predictions = debug_predictions.astype(int)


print("\n" + "=" * 60)
print("TRUE CLASS DISTRIBUTION")
print("=" * 60)

print(y_debug.value_counts().sort_index())


print("\n" + "=" * 60)
print("PREDICTED CLASS DISTRIBUTION")
print("=" * 60)

print(pd.Series(debug_predictions)
      .value_counts()
      .sort_index())


print("\nNumber of unique predicted classes:")
print(len(np.unique(debug_predictions)))

print("\nPredicted classes:")
print(sorted(np.unique(debug_predictions)))


print("\n" + "=" * 60)
print("BALANCED DEBUG DATASET ACCURACY")
print("=" * 60)

debug_accuracy = accuracy_score(
    y_debug,
    debug_predictions
)

print("\nAccuracy:", debug_accuracy)

print("\nCorrect predictions:",
      (y_debug.to_numpy() == debug_predictions).sum())

print("\nTotal samples:",
      len(y_debug))

BALANCED DEBUGGING PREDICTION TEST

Number of predictions:
4923

Number of debugging samples:
4923

Predictions aligned:
True

TRUE CLASS DISTRIBUTION
Label_Encoded
0     300
1     300
2     300
3     300
4     300
5     300
6     300
7     300
8     300
9     300
10    300
11    300
12    300
13      9
14    300
15     29
16    300
17     85
18    300
Name: count, dtype: int64

PREDICTED CLASS DISTRIBUTION
0     4796
2        9
3       25
4        2
5       18
6        2
7        5
8        1
9        4
10       9
11       2
12       6
13       2
14       5
15       6
16       6
17      23
18       2
Name: count, dtype: int64

Number of unique predicted classes:
18

Predicted classes:
[np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]

BALANCED DEBUG DATASET ACCURACY

Accuracy: 0.086735730245785

# Documentation: Balanced Debugging Dataset and Model Collapse Investigation

After fixing the earlier feature–label row alignment and 19-class compatibility problems, the next objective was to determine whether the poor model performance was caused by the test dataset or by the trained model itself.

A controlled debugging dataset was therefore created from the training data. Initially, the dataset attempted to select 300 samples from every class. However, the dataset contains several extremely rare classes. Classes 13, 15, and 17 contained only 9, 29, and 85 training samples respectively. Because these classes did not have 300 available samples, they were unintentionally excluded, resulting in a debugging dataset containing only 16 classes.

The debugging dataset creation process was corrected by selecting the number of samples using:

min(300, available_samples)

This ensured that every available class was included. Classes with at least 300 samples contributed 300 samples, while the rare classes contributed all their available samples.

The corrected debugging dataset contained 4,923 samples and 77 features. Feature rows and label rows were successfully aligned, and all 19 classes (0–18) were present.

The trained 19-class XGBoost model was then used to generate predictions on this balanced debugging dataset. Despite the dataset containing examples from every class, the model predicted Class 0 for 4,796 out of 4,923 samples, representing approximately 97.4% of all predictions. The remaining predictions were distributed sparsely among the other classes.

The model achieved an accuracy of approximately 8.67%, with only 427 correct predictions out of 4,923 samples.

This behaviour was consistent with the earlier controlled test evaluation and training evaluation experiments, both of which also produced very low accuracy and a strong tendency to predict Class 0.

Therefore, the debugging process established that the issue is not caused by missing classes, feature-label misalignment, or the test dataset alone. The evidence indicates that the trained model is strongly biased toward predicting Class 0. The next stage is to inspect the internal XGBoost multiclass configuration and probability outputs to determine whether the collapse is caused by class imbalance during training, model configuration, objective behaviour, or probability interpretation.

### Verify XGB objective

In [44]:
print("=" * 60)
print("XGBOOST MULTICLASS CONFIGURATION CHECK")
print("=" * 60)

# Check model parameters
params = xgb_model_19.get_params()

print("\nObjective:")
print(params.get("objective"))

print("\nNumber of classes:")
print(params.get("num_class"))

print("\nEvaluation metric:")
print(params.get("eval_metric"))

print("\nNumber of estimators:")
print(params.get("n_estimators"))

print("\nLearning rate:")
print(params.get("learning_rate"))


print("\n" + "=" * 60)
print("MODEL INTERNAL CLASS CHECK")
print("=" * 60)

print("\nModel classes:")
print(xgb_model_19.classes_)

print("\nNumber of model classes:")
print(len(xgb_model_19.classes_))


print("\n" + "=" * 60)
print("PROBABILITY OUTPUT CHECK")
print("=" * 60)

# Predict probabilities
debug_probabilities = xgb_model_19.predict_proba(X_debug)

print("\nProbability matrix shape:")
print(debug_probabilities.shape)

print("\nExpected shape:")
print("(number_of_samples, number_of_classes)")

print("\nActual number of samples:")
print(debug_probabilities.shape[0])

print("\nActual number of classes:")
print(debug_probabilities.shape[1])


# Check probability sums
probability_sums = debug_probabilities.sum(axis=1)

print("\nFirst 10 probability sums:")
print(probability_sums[:10])

print("\nMinimum probability sum:")
print(probability_sums.min())

print("\nMaximum probability sum:")
print(probability_sums.max())

XGBOOST MULTICLASS CONFIGURATION CHECK

Objective:
multi:softprob

Number of classes:
19

Evaluation metric:
mlogloss

Number of estimators:
200

Learning rate:
0.1

MODEL INTERNAL CLASS CHECK

Model classes:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]

Number of model classes:
19

PROBABILITY OUTPUT CHECK

Probability matrix shape:
(4923, 19)

Expected shape:
(number_of_samples, number_of_classes)

Actual number of samples:
4923

Actual number of classes:
19

First 10 probability sums:
[1.         0.99999994 1.         1.         0.9999999  1.
 0.99999994 0.9999999  1.0000001  0.99999994]

Minimum probability sum:
0.9999998

Maximum probability sum:
1.0000001


In [45]:
print("=" * 60)
print("PREDICT VS PREDICT_PROBA CHECK")
print("=" * 60)

# Class with highest probability
proba_predictions = np.argmax(
    debug_probabilities,
    axis=1
)

# Normal model predictions
normal_predictions = xgb_model_19.predict(
    X_debug
).astype(int)


print("\nFirst 20 predictions from predict():")
print(normal_predictions[:20])

print("\nFirst 20 predictions from predict_proba argmax:")
print(proba_predictions[:20])


print("\nPredictions identical:")
print(
    np.array_equal(
        normal_predictions,
        proba_predictions
    )
)


print("\n" + "=" * 60)
print("ARGMAX PREDICTION DISTRIBUTION")
print("=" * 60)

print(
    pd.Series(proba_predictions)
    .value_counts()
    .sort_index()
)

PREDICT VS PREDICT_PROBA CHECK

First 20 predictions from predict():
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

First 20 predictions from predict_proba argmax:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Predictions identical:
True

ARGMAX PREDICTION DISTRIBUTION
0     4796
2        9
3       25
4        2
5       18
6        2
7        5
8        1
9        4
10       9
11       2
12       6
13       2
14       5
15       6
16       6
17      23
18       2
Name: count, dtype: int64


### For samples belonging to each true class, what probabilities does the model assign to the 19 classes?

In [46]:
print("=" * 70)
print("CLASS-WISE PROBABILITY ANALYSIS")
print("=" * 70)

# Create DataFrame from probabilities
probability_df = pd.DataFrame(
    debug_probabilities,
    columns=[f"Class_{i}" for i in range(19)]
)

# Add true labels
probability_df["True_Label"] = y_debug.values


# Calculate average probability for each true class
class_probability_analysis = (
    probability_df
    .groupby("True_Label")
    .mean()
)


print("\nAverage predicted probabilities by TRUE class:\n")

pd.set_option("display.max_columns", None)

print(class_probability_analysis.round(4))

CLASS-WISE PROBABILITY ANALYSIS

Average predicted probabilities by TRUE class:

            Class_0  Class_1  Class_2  Class_3  Class_4  Class_5  Class_6  \
True_Label                                                                  
0            0.3066   0.0743   0.0023   0.0009   0.0750   0.0020   0.0740   
1            0.2977   0.0844   0.0022   0.0010   0.0743   0.0019   0.0743   
2            0.2915   0.0725   0.0313   0.0009   0.0728   0.0019   0.0726   
3            0.2825   0.0715   0.0022   0.0560   0.0711   0.0021   0.0710   
4            0.2971   0.0737   0.0023   0.0010   0.0862   0.0019   0.0728   
5            0.2892   0.0722   0.0024   0.0010   0.0714   0.0418   0.0718   
6            0.2954   0.0743   0.0024   0.0009   0.0747   0.0018   0.0875   
7            0.2956   0.0740   0.0023   0.0009   0.0739   0.0017   0.0739   
8            0.2955   0.0745   0.0024   0.0010   0.0750   0.0021   0.0734   
9            0.2960   0.0739   0.0025   0.0011   0.0733   0.0020   0.074

### Find the model's preferred prediction for each true class

In [47]:
print("\n" + "=" * 70)
print("MOST LIKELY PREDICTION FOR EACH TRUE CLASS")
print("=" * 70)

for true_class in sorted(y_debug.unique()):

    # Get rows belonging to this true class
    class_mask = y_debug.values == true_class

    # Get probabilities for those rows
    class_probabilities = debug_probabilities[class_mask]

    # Average probabilities
    average_probabilities = class_probabilities.mean(axis=0)

    # Find highest probability class
    predicted_class = np.argmax(average_probabilities)

    print(
        f"\nTrue Class {true_class} "
        f"--> Model prefers Class {predicted_class}"
    )

    print(
        "Highest average probability:",
        round(average_probabilities[predicted_class], 6)
    )

    print(
        "Probability assigned to correct class:",
        round(average_probabilities[true_class], 6)
    )


MOST LIKELY PREDICTION FOR EACH TRUE CLASS

True Class 0 --> Model prefers Class 0
Highest average probability: 0.306562
Probability assigned to correct class: 0.306562

True Class 1 --> Model prefers Class 0
Highest average probability: 0.297651
Probability assigned to correct class: 0.084416

True Class 2 --> Model prefers Class 0
Highest average probability: 0.29147
Probability assigned to correct class: 0.031296

True Class 3 --> Model prefers Class 0
Highest average probability: 0.282506
Probability assigned to correct class: 0.055953

True Class 4 --> Model prefers Class 0
Highest average probability: 0.297088
Probability assigned to correct class: 0.086226

True Class 5 --> Model prefers Class 0
Highest average probability: 0.289235
Probability assigned to correct class: 0.041811

True Class 6 --> Model prefers Class 0
Highest average probability: 0.295351
Probability assigned to correct class: 0.087548

True Class 7 --> Model prefers Class 0
Highest average probability: 0.2956

### Confusion matrix

In [48]:
print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_debug,
    debug_predictions,
    labels=list(range(19))
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{i}" for i in range(19)],
    columns=[f"Pred_{i}" for i in range(19)]
)

print(cm_df)


CONFUSION MATRIX
         Pred_0  Pred_1  Pred_2  Pred_3  Pred_4  Pred_5  Pred_6  Pred_7  \
True_0      300       0       0       0       0       0       0       0   
True_1      300       0       0       0       0       0       0       0   
True_2      291       0       9       0       0       0       0       0   
True_3      275       0       0      25       0       0       0       0   
True_4      298       0       0       0       2       0       0       0   
True_5      282       0       0       0       0      18       0       0   
True_6      298       0       0       0       0       0       2       0   
True_7      295       0       0       0       0       0       0       5   
True_8      299       0       0       0       0       0       0       0   
True_9      296       0       0       0       0       0       0       0   
True_10     291       0       0       0       0       0       0       0   
True_11     298       0       0       0       0       0       0       0   
True_12